In [4]:
import os
from google.cloud import bigquery
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, to_timestamp, avg, first)

#Authentification
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = r"C:\Users\USER\Downloads\ecommerce-analytics-488100-540b9e9c99e9.json"
# Initialize BigQuery client
client = bigquery.Client()
print(client.project)


gcs_connector_jar = r"C:\spark_jars\gcs-connector-latest-hadoop2.jar"

spark = (
    SparkSession.builder
    .appName("eadp-silver")
    # JAR du connecteur GCS
    .config("spark.jars", gcs_connector_jar)
    # Config pour Windows / Hadoop
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", r"C:\Users\USER\Downloads\ecommerce-analytics-488100-540b9e9c99e9.json")
    # Optionnel : temp dir pour Hadoop sur Windows
    .config("spark.local.dir", r"C:\spark_tmp")
    .getOrCreate()
)

# Load bronze data from CSV files, then transform to silver
customers = spark.read.option("Header", True).csv("../bronze/customers_dataset.csv")
silver_customers = (
    customers
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        col("customer_city").alias("city"),
        col("customer_state").alias("state")
    )
    .dropDuplicates(["customer_id"])
)

geo = spark.read.option("header", True).csv("../bronze/geolocation_dataset.csv")
silver_geo = (
    geo
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("latitude"),
        avg("geolocation_lng").alias("longitude"),
        first("geolocation_city").alias("geo_city"),
        first("geolocation_state").alias("geo_state")
    )
)

silver_customers = (
    silver_customers
    .join(
        silver_geo,
        silver_customers.customer_zip_code_prefix
        == silver_geo.geolocation_zip_code_prefix,
        "left"
    )
    .drop("customer_zip_code_prefix", "geolocation_zip_code_prefix")
)

orders = spark.read.option("header", True).csv("../bronze/orders_dataset.csv")
silver_orders = (
    orders
    .select(
        "order_id",
        "customer_id",
        "order_status",
        to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        to_timestamp("order_delivered_customer_date").alias("delivered_ts"),
        to_timestamp("order_estimated_delivery_date").alias("estimated_delivery_ts")
    )
    .dropDuplicates(["order_id"])
)

items = spark.read.option("header", True).csv("../bronze/order_items_dataset.csv")
silver_order_items = (
    items
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        col("price").cast("double"),
        col("freight_value").cast("double"),
        to_timestamp("shipping_limit_date").alias("shipping_limit_ts")
    )
)

payments = spark.read.option("header", True).csv("../bronze/order_payments_dataset.csv")
silver_payments = (
    payments
    .groupBy("order_id")
    .agg(
        avg("payment_value").alias("payment_value"),
        first("payment_type").alias("payment_type"),
        first("payment_installments").alias("payment_installments")
    )
)

reviews = spark.read.option("header", True).csv("../bronze/order_reviews_dataset.csv")
silver_reviews = (
    reviews
    .select(
        "order_id",
        col("review_score").cast("int")
    )
    .dropDuplicates(["order_id"])
)

products = spark.read.option("header", True).csv("../bronze/products_dataset.csv")
categories = spark.read.option("header", True).csv("../bronze/product_category_name_translation.csv")
silver_products = (
    products
    .join(
        categories,
        "product_category_name",
        "left"
    )
    .select(
        "product_id",
        col("product_category_name_english").alias("category"),
        col("product_weight_g").cast("double"),
        col("product_length_cm").cast("double"),
        col("product_height_cm").cast("double"),
        col("product_width_cm").cast("double")
    )
    .dropDuplicates(["product_id"])
)

sellers = spark.read.option("header", True).csv("../bronze/sellers_dataset.csv")
silver_sellers = (
    sellers
    .join(
        silver_geo,
        sellers.seller_zip_code_prefix
        == silver_geo.geolocation_zip_code_prefix,
        "left"
    )
    .select(
        "seller_id",
        col("seller_city").alias("city"),
        col("seller_state").alias("state"),
        "latitude",
        "longitude"
    )
    .dropDuplicates(["seller_id"])
)

# Write silver data to GCS in Parquet format

silver_customers.write.mode("overwrite").parquet("gs://ecommerce-analytics-bronze-akabi/silver/customers")
silver_orders.write.mode("overwrite").parquet("gs://ecommerce-analytics-bronze-akabi/silver/orders")
silver_order_items.write.mode("overwrite").parquet("gs://ecommerce-analytics-bronze-akabi/silver/order_items")
silver_payments.write.mode("overwrite").parquet("gs://ecommerce-analytics-bronze-akabi/silver/payments")
silver_reviews.write.mode("overwrite").parquet("gs://ecommerce-analytics-bronze-akabi/silver/reviews")
silver_products.write.mode("overwrite").parquet("gs://ecommerce-analytics-bronze-akabi/silver/products")
silver_sellers.write.mode("overwrite").parquet("gs://ecommerce-analytics-bronze-akabi/silver/sellers")

ecommerce-analytics-488100


In [5]:
# Start By creating BigQuery Dataset, the tables will be crated automatically
# In this code, the dataset is named ecommerce_silver;
# Change the name in the table_id if yours is different.

tables = ["customers", "orders", "order_items", "payments", "reviews", "products", "sellers"]

# Load data into BigQuery

for table in tables:
#Change the GoogleProjectID to match yours
    table_id = f"ecommerce-analytics-488100.ecommerce_silver.{table}"
    uri = f"gs://ecommerce-analytics-bronze-akabi/silver/{table}/*"
    
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition="WRITE_TRUNCATE"  # overwrite if table exists
    )
    
    load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
    load_job.result()
    
    print(f"Loaded {table} successfully!")

Loaded customers successfully!
Loaded orders successfully!
Loaded order_items successfully!
Loaded payments successfully!
Loaded reviews successfully!
Loaded products successfully!
Loaded sellers successfully!
